# 04 — Modélisation segmentation client

Objectifs :
- comparer K-Means, Agglomerative et GMM (k = 3 à 6) ;
- appliquer la méthode du coude (Elbow) ;
- visualiser les clusters en PCA 2D ;
- interpréter les profils métier.

Le pipeline de production est dans `src/models/train_clustering_model.py`.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CLUSTER_DIR = ROOT / "models" / "clustering"
sns.set_theme(style="whitegrid")

## 1. Comparaison des algorithmes

In [ ]:
comparison = pd.read_csv(CLUSTER_DIR / "clustering_model_comparison.csv", index_col=0)
best = json.loads((CLUSTER_DIR / "clustering_best_model.json").read_text(encoding="utf-8"))
comparison[["silhouette", "davies_bouldin", "calinski_harabasz", "n_clusters"]].sort_values(
    "silhouette", ascending=False
)

In [ ]:
pivot = comparison.reset_index().assign(
    algo=lambda d: d["index"].str.extract(r"^(\w+)_k"),
    k=lambda d: d["index"].str.extract(r"_k(\d+)").astype(int),
)
fig, ax = plt.subplots(figsize=(10, 4))
for algo, group in pivot.groupby("algo"):
    ax.plot(group["k"], group["silhouette"], marker="o", label=algo)
ax.axvline(4, color="crimson", linestyle="--", label="k retenu = 4")
ax.set_title("Silhouette par algorithme et par k")
ax.set_xlabel("Nombre de clusters")
ax.set_ylabel("Silhouette")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Elbow Method (K-Means)

In [ ]:
elbow_path = CLUSTER_DIR / "clustering_elbow.csv"
if elbow_path.exists():
    elbow = pd.read_csv(elbow_path)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(elbow["k"], elbow["inertia"], marker="o", color="#157f7f")
    ax.axvline(4, color="crimson", linestyle="--", label="k retenu = 4")
    ax.set_title("Elbow Method — inertie vs k")
    ax.set_xlabel("k")
    ax.set_ylabel("Inertie")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Lancer : python -m src.models.train_clustering_model")

## 3. Projection PCA 2D des clusters retenus

## 2bis. DBSCAN (hors modèle retenu)

In [ ]:
dbscan_path = CLUSTER_DIR / "clustering_dbscan_comparison.csv"
if dbscan_path.exists():
    dbscan = pd.read_csv(dbscan_path, index_col=0)
    evaluated = dbscan[dbscan.get("status", "") == "evaluated"].sort_values("silhouette", ascending=False)
    display(evaluated.head(8)[["silhouette", "davies_bouldin", "n_clusters", "noise_ratio"]])
    print("DBSCAN écarté : trop de bruit et pas de predict() pour l'API.")
else:
    print("Lancer : python -m src.models.train_clustering_model")

In [ ]:
from src.data.preprocessing import preprocess_cluster
from src.features.build_features import build_cluster_features

raw = pd.read_csv(ROOT / "data" / "raw" / "data_cluster.csv", sep=";")
model = joblib.load(CLUSTER_DIR / "cluster_model.joblib")
prepared = build_cluster_features(preprocess_cluster(raw))
model_input = prepared.drop(
    columns=[c for c in ("ID", "Response", "Complain", "Dt_Customer") if c in prepared.columns]
)
model_input = model_input.reindex(columns=list(model.named_steps["preprocessor"].feature_names_in_))
transformed = model.named_steps["preprocessor"].transform(model_input)
labels = model.named_steps["clustering"].predict(transformed)
coords = PCA(n_components=2, random_state=42).fit_transform(transformed)

plot_df = pd.DataFrame({"pc1": coords[:, 0], "pc2": coords[:, 1], "cluster": labels})
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(data=plot_df, x="pc1", y="pc2", hue="cluster", palette="tab10", alpha=0.7, ax=ax)
ax.set_title(f"PCA 2D — modèle retenu : {best['best_model']}")
plt.tight_layout()
plt.show()

## 4. Profils métier par cluster

In [ ]:
profiles = pd.read_csv(CLUSTER_DIR / "cluster_profiles.csv")
profiles

In [ ]:
heat_cols = [
    "Income", "Total_Spending", "NumWebPurchases", "NumStorePurchases",
    "NumDealsPurchases", "Campaign_Acceptance_Total", "cluster_size",
]
heat_cols = [c for c in heat_cols if c in profiles.columns]
norm = profiles.set_index("cluster")[heat_cols]
norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-9)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(norm.T, annot=True, fmt=".2f", cmap="viridis", ax=ax)
ax.set_title("Profils normalisés par cluster")
plt.tight_layout()
plt.show()

## Interprétation

- **GMM k=4** combine un score silhouette modéré et des personas lisibles.
- L'**Elbow** et la silhouette convergent vers k = 4 comme compromis lisible.
- **Cluster 0** : dormants à faible valeur — réactivation.
- **Cluster 1** : promo digitaux (42 clients) — segment petit, à valider par campagne test.
- **Cluster 2** : premium ultra-engagés — programme VIP.
- **Cluster 3** : fidèles à forte valeur — fidélisation et cross-sell.

> La PCA est utilisée ici à des fins de **visualisation** ; elle n'entre pas dans le pipeline de clustering final.

## Reproduction

```bash
python -m src.models.train_clustering_model
python -m src.visualization.generate_report_figures
```